# 01 - Model bake-off (Niti-Agent)
**Goal:** pick the smallest open model that handles our tasks on Nepali text. Do NOT choose by reputation; measure.

Before running: Runtime > Change runtime type > **T4 GPU**.

Tasks tested: (1) extraction to JSON, (2) Nepali explanation, (3) messy-question understanding.
Fill `TEST_CASES` with REAL examples from your collected documents.

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Candidates: verify these IDs exist on huggingface.co before running.
# Some models (Llama, Gemma) are gated: accept the license on HF and set HF_TOKEN.
CANDIDATES = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    # add more here after checking they fit on a T4 in 4-bit
]

SCHEMA_HINT = """Return ONLY valid JSON with keys:
title_en, office, mode (online|in_person|both), steps (list of {order, action}),
documents (list of {name, required (true/false), reason}), fee, time_needed."""

# TODO: replace with 20 REAL cases. type = extract | explain | messy
TEST_CASES = [
    {"type": "extract", "input": "PASTE REAL NEPALI/ENGLISH NOTICE TEXT HERE", "expected": {}},
]

In [ ]:
def load(model_id):
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=q, device_map="auto")
    return tok, model

def generate(tok, model, prompt, max_new_tokens=512):
    msgs = [{"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def valid_json(text):
    try:
        json.loads(text[text.index("{"): text.rindex("}") + 1]); return True
    except Exception:
        return False

In [ ]:
results = {}
for mid in CANDIDATES:
    tok, model = load(mid)
    rows = []
    for case in TEST_CASES:
        prompt = f"{SCHEMA_HINT}\n\nText:\n{case['input']}" if case["type"] == "extract" else case["input"]
        t0 = time.time(); out = generate(tok, model, prompt)
        rows.append({"type": case["type"], "seconds": round(time.time() - t0, 1),
                     "valid_json": valid_json(out) if case["type"] == "extract" else None, "output": out})
    results[mid] = rows
    del model; torch.cuda.empty_cache()

# Save so a disconnect does not lose your work
from google.colab import drive; drive.mount("/content/drive")
json.dump(results, open("/content/drive/MyDrive/niti_bakeoff.json", "w"), ensure_ascii=False, indent=2)
for mid, rows in results.items():
    ext = [r for r in rows if r["valid_json"] is not None]
    print(mid, "valid JSON:", sum(r["valid_json"] for r in ext), "/", len(ext))

## After running
1. Read the outputs yourself. Rate Nepali quality 1-5 per case in a spreadsheet.
2. Pick the smallest model with acceptable valid-JSON rate AND Nepali quality.
3. Record the table in `eval/` - it goes in your final write-up.